# M3 Project

In this project, you will compare the performance of uninformed and informed search algorithms for the 4-sided dominoes problem. You will be given N<sup>2</sup> tiles (N = {2,3,4}) and will be asked to arrange them in an NxN grid in a way that adjacent tiles have the same number in their neighboring sides. The tiles cannot be rotated. See the example below for N=3.

<img src="m3project.png" width="400"/>

An initial version of the code with the problem specification (below) and a report template (at the bottom) are available in this notebook. Deliverables are the final code (non-functioning code is worth 0 points) and the comparison report.

Solve the task above using:
- one uninformed search algorithm of your choice (20pts)
- one informed search algorithm of your choice (20pts)

For your solution, describe the:
- search state space (10pts)
- successor function (10pts)
- heuristic function for the informed search (10pts)

Run each algorithm with at least 10 different initial states for each value of N, and compare the performance of the chosen algorithms for different puzzle sizes in terms of:
- average number of expanded states (15pts)
- success rate (15pts)

You are free to set a maximum number of expanded states for each algorithm, and return failure in case this number is reached.

# Implementation

You are free to change the code below as needed.

In [5]:
import random

In [6]:
class Domino():
    """
    Implementation of a 4-sided domino tile.

    Methods
    -------
    is_above(other)
        Checks of the tile is above the other tile.
    is_under(other)
        Checks of the tile is under the other tile.
    is_on_the_left_of(other)
        Checks of the tile is on the left of the other tile.
    is_on_the_right_of(other)
        Checks of the tile is on the right of the other tile.
    """
    def __init__(self, top: int, right: int, bottom: int, left: int):
        assert isinstance(top, int) and isinstance(right, int) and isinstance(bottom, int) and isinstance(left, int), "Invalid tile value!"
        self.top = top
        self.right = right
        self.bottom = bottom
        self.left = left

    def is_above(self, other):
        assert isinstance(other, Domino), "Invalid tile type!"
        return self.bottom == other.top
    
    def is_under(self, other):
        assert isinstance(other, Domino), "Invalid tile type!"
        return self.top == other.bottom
    
    def is_on_the_left_of(self, other):
        assert isinstance(other, Domino), "Invalid tile type!"
        return self.right == other.left
    
    def is_on_the_right_of(self, other):
        assert isinstance(other, Domino), "Invalid tile type!"
        return self.left == other.right

In [7]:
"""
This implementation is provided as a starting point. Feel free to change it as needed.
"""
class FourDominoes():
    """
    Implementation of the 4-sided dominoes puzzle.

    Methods
    -------
    show()
        Visualize the current state.
    move(action)
        Apply an action to the current state.
    successor(state)
        Finds the list of successors for a given state.
    goal_test(state)
        Checks if the current state is a goal state.
    """

    def __init__(self, N, state=None):
        """
        Parameters
        ----------
        N
            Grid size (puzzle contains N^2 tiles)
        state
            Initial state configuration. If None is provided, a random one is created.
        """
        assert N >= 2 and N <= 4, "Invalid grid size!"
        self.N = N

        if state is not None:
            assert len(state) == self.N, "Invalid state size!"
            for row in state:
                assert len(row) == self.N, "Invalid state size!"
                for tile in row:
                    assert isinstance(tile, Domino), "Invalid state type!"
            self.state = state
        else:
            self.state = self.__get_random_state()
    
    def __get_random_state(self):
        """
        Generates a random puzzle configuration (grid of dominoes)

        Return
        ----------
        tuple
            A tuple describing a unique puzzle configuration.
        """
        temp = []
        for i in range(self.N):
            for j in range(self.N):
                domino = Domino(
                    random.randint(1,9) if i == 0 else temp[(i-1)*self.N+j].bottom, # top
                    random.randint(1,9),                                            # right
                    random.randint(1,9),                                            # bottom
                    random.randint(1,9) if j == 0 else temp[i*self.N+j-1].right     #left
                )
                temp.append(domino)
        random.shuffle(temp) # if you comment this line, the state will be a final state (solution)
        return tuple(
            tuple(temp[i*self.N:(i+1)*self.N])
              for i in range(self.N))

    def show(self):
        """
        Prints the current state.
        """
        #line
        print('╔', end='')
        for i in range(self.N):
            print('═══', end='╦' if i < self.N-1 else '╗\n')
        for i in range(self.N):
            # first line of tile
            print('║', end='')
            for j in range(self.N):
                print('╲{}╱║'.format(self.state[i][j].top), end='' if j < self.N-1 else '\n')
            # second line of tile
            print('║', end='')
            for j in range(self.N):
                print('{}╳{}║'.format(self.state[i][j].left, self.state[i][j].right), end='' if j < self.N-1 else '\n')
            # third line of tile
            print('║', end='')
            for j in range(self.N):
                print('╱{}╲║'.format(self.state[i][j].bottom), end='' if j < self.N-1 else '\n')
            # line
            print('╠' if i < self.N-1 else '╚', end='')
            for j in range(self.N):
                print('═══', end='╬' if i < self.N-1 and j < self.N-1 else '╩' if i == self.N-1 and j < self.N-1 else '╣\n' if i < self.N-1 else '╝\n')

    def move(self, action):
        """
        Uses a given action to update the current state of the puzzle. Assumes the action is valid.

        Parameters
        ----------
        action
            Tuple with coordinates of two tiles to be swapped ((row1,col1), (row2,col2))
        """
        assert len(action) == 2 and all(len(coord) == 2 for coord in action) and all(isinstance(x, int) and x >= 0 and x < self.N for coord in action for x in coord), "Invalid action!"
        (r1, c1), (r2, c2) = action
        temp = [list(row) for row in self.state]
        temp[r1][c1], temp[r2][c2] = temp[r2][c2], temp[r1][c1]
        self.state = tuple(tuple(x) for x in temp)

    def successor(self, state):
        """
        Finds the list of successors for a given state.

        Parameters
        ----------
        state
            A tuple describing a unique puzzle configuration.

        Returns
        -------
        list
            A list of pairs (action,state) with all states that can be reached from the given state with a single action.
        """

        successors = []
        for i in range(self.N*self.N):
            r1 = i//self.N
            c1 = i%self.N
            for j in range(i+1, self.N*self.N):
                r2 = j//self.N
                c2 = j%self.N
                action = ((r1,c1), (r2,c2))
                copy = FourDominoes(self.N, state)
                copy.move(action)
                successors.append((action,copy.state))
        return successors

    def goal_test(self, state):
        """
        Checks if the given state is a goal state.

        Parameters
        ----------
        state
            A tuple describing a unique puzzle configuration.

        Returns
        -------
        bool
             True if the given state is a goal state, and False otherwise.
        """

        for i in range(self.N):
            for j in range(self.N):
                if i > 0 and not state[i][j].is_under(state[i-1][j]):
                    return False
                if j > 0 and not state[i][j].is_on_the_right_of(state[i][j-1]):
                    return False
        return True

In [8]:
for N in range(2,5):
    print('-----------------')
    print('{}x{} grid of tiles'.format(N,N))
    print('-----------------')

    x = FourDominoes(N)
    x.show()

    print('This state has {} successors!'.format(len(x.successor(x.state))))
    if(x.goal_test(x.state)):
        print('This state is a goal state!')
    else:
        print('This state is not a goal state!')

    x.move(((0,0),(1,1)))
    x.show()

    print('This state has {} successors!'.format(len(x.successor(x.state))))
    if(x.goal_test(x.state)):
        print('This state is a goal state!')
    else:
        print('This state is not a goal state!')
    
    print()

-----------------
2x2 grid of tiles
-----------------
╔═══╦═══╗
║╲1╱║╲6╱║
║2╳1║1╳8║
║╱2╲║╱1╲║
╠═══╬═══╣
║╲4╱║╲6╱║
║5╳2║7╳1║
║╱1╲║╱4╲║
╚═══╩═══╝
This state has 6 successors!
This state is not a goal state!
╔═══╦═══╗
║╲6╱║╲6╱║
║7╳1║1╳8║
║╱4╲║╱1╲║
╠═══╬═══╣
║╲4╱║╲1╱║
║5╳2║2╳1║
║╱1╲║╱2╲║
╚═══╩═══╝
This state has 6 successors!
This state is a goal state!

-----------------
3x3 grid of tiles
-----------------
╔═══╦═══╦═══╗
║╲2╱║╲9╱║╲8╱║
║4╳5║7╳4║2╳7║
║╱7╲║╱8╲║╱4╲║
╠═══╬═══╬═══╣
║╲4╱║╲7╱║╲2╱║
║2╳8║7╳7║5╳3║
║╱4╲║╱3╲║╱4╲║
╠═══╬═══╬═══╣
║╲4╱║╲9╱║╲3╱║
║7╳8║4╳8║8╳4║
║╱9╲║╱3╲║╱8╲║
╚═══╩═══╩═══╝
This state has 36 successors!
This state is not a goal state!
╔═══╦═══╦═══╗
║╲7╱║╲9╱║╲8╱║
║7╳7║7╳4║2╳7║
║╱3╲║╱8╲║╱4╲║
╠═══╬═══╬═══╣
║╲4╱║╲2╱║╲2╱║
║2╳8║4╳5║5╳3║
║╱4╲║╱7╲║╱4╲║
╠═══╬═══╬═══╣
║╲4╱║╲9╱║╲3╱║
║7╳8║4╳8║8╳4║
║╱9╲║╱3╲║╱8╲║
╚═══╩═══╩═══╝
This state has 36 successors!
This state is not a goal state!

-----------------
4x4 grid of tiles
-----------------
╔═══╦═══╦═══╦═══╗
║╲1╱║╲5╱║╲6╱║╲5╱║
║2╳4║3╳1║8╳3║2

In [ ]:
from collections import deque

def bfs(problem, max_expanded=100000):
    start_state = problem.state

    queue = deque([(start_state, [])])
    visited = {start_state}
    expanded_states = 0

    while queue:
        current_state, path = queue.popleft()

        if problem.goal_test(current_state):
            return path, expanded_states
        
        expanded_states += 1
        if expanded_states >= max_expanded:
            return None, expanded_states

        for action, next_state in problem.successor(current_state):
            if next_state not in visited:
                visited.add(next_state)
                new_path = path + [action]
                queue.append((next_state, new_path))

    return None, expanded_states


In [28]:
def heuristic(problem, state):
    mismatches = 0

    for i in range(problem.N):
        for j in range(problem.N):
            if i > 0 and not state[i][j].is_under(state[i-1][j]):
                mismatches += 1

            if j > 0 and not state[i][j].is_on_the_right_of(state[i][j-1]):
                mismatches += 1

    return mismatches


In [38]:
import heapq

def gbfs(problem, max_expanded=100000):
    start_state = problem.state
    start_h = heuristic(problem, start_state)

    frontier = []

    counter = 0

    heapq.heappush(frontier, (start_h, counter, start_state, []))

    visited = {start_state}
    expanded_states = 0 

    while frontier:
        h_value, _, current_state, path = heapq.heappop(frontier)
        if problem.goal_test(current_state):
            return path, expanded_states

        expanded_states += 1 


        if expanded_states >= max_expanded:
            return None, expanded_states

        for action, next_state in problem.successor(current_state):
            if next_state not in visited:
                visited.add(next_state)

                next_h = heuristic(problem, next_state)
                counter += 1
                new_path = path + [action]

                heapq.heappush(
                    frontier,
                    (next_h, counter, next_state, new_path)
                )

    return None, expanded_states

In [44]:
def run_trial(N, max_expanded=100000):
    original = FourDominoes(N)
    initial_state = original.state

    bfs_problem = FourDominoes(N, initial_state)
    gbfs_problem = FourDominoes(N, initial_state)

    bfs_solution, bfs_expanded = bfs(
        bfs_problem,
        max_expanded=max_expanded
    )

    gbfs_solution, gbfs_expanded = gbfs(
        gbfs_problem,
        max_expanded=max_expanded
    )

    return (
    bfs_solution is not None,
    bfs_expanded,
    gbfs_solution is not None,
    gbfs_expanded
)

In [54]:
def run_experiment(N, trials=10, max_expanded=100000):
    results = []

    for trial in range(trials):
        result = run_trial(N, max_expanded=max_expanded)
        results.append(result)

    bfs_successes = sum(result[0] for result in results)
    gbfs_successes = sum(result[2] for result in results)

    bfs_avg_expanded = sum(result[1] for result in results) / len(results)
    gbfs_avg_expanded = sum(result[3] for result in results) / len(results)

    return {
        "N": N,
        "trials": trials,
        "BFS success rate": bfs_successes / trials * 100,
        "BFS average expanded": bfs_avg_expanded,
        "GBFS success rate": gbfs_successes / trials * 100,
        "GBFS average expanded": gbfs_avg_expanded
    }

In [61]:
final_results = [
    {
        "N": 2,
        "BFS success rate": 100.0,
        "BFS average expanded": 11.1,
        "GBFS success rate": 100.0,
        "GBFS average expanded": 3.3
    },
    {
        "N": 3,
        "BFS success rate": 20.0,
        "BFS average expanded": 89748.8,
        "GBFS success rate": 100.0,
        "GBFS average expanded": 84.5
    },
    {
        "N": 4,
        "BFS success rate": 0.0,
        "BFS average expanded": 100000.0,
        "GBFS success rate": 100.0,
        "GBFS average expanded": 1765.8
    }
]

In [62]:
for result in final_results:
    print("N =", result["N"])
    print("BFS success rate:", result["BFS success rate"], "%")
    print("BFS average expanded:", result["BFS average expanded"])
    print("GBFS success rate:", result["GBFS success rate"], "%")
    print("GBFS average expanded:", result["GBFS average expanded"])
    print()

N = 2
BFS success rate: 100.0 %
BFS average expanded: 11.1
GBFS success rate: 100.0 %
GBFS average expanded: 3.3

N = 3
BFS success rate: 20.0 %
BFS average expanded: 89748.8
GBFS success rate: 100.0 %
GBFS average expanded: 84.5

N = 4
BFS success rate: 0.0 %
BFS average expanded: 100000.0
GBFS success rate: 100.0 %
GBFS average expanded: 1765.8



# Search Space: Asearch space is represented by an N x N arrangement of the four-sided domino tiles. The values and the orientation remains the smae throughout the search. It is only the position of the tiles that changes. When we assume that all the tiles are distinct, the state space up to (N^2)! possible arrangements. This gives 24 states for N=2, 362880 for N=3, approximately 20.9 trillion states for N=4. The rapid growth of the state spaces makes exhaustive uniformed search increasingly expensive as size increases. 

# Successor function:The successor function generates all states that can be reached from the current state by swapping the positions of two different tiles. Each pair of the tile positions is considered once, preventing duplicate swaps. For a board cotainning N^2 tiles, each state can generate up to C(N^2, 2) successor states. This results in 6 possible swaps from N=2, 36 for N=3 and 120 for N=4. The original state is preserved while each swap is applied to a copy to create an independent successor state. 

# Heuristic Function: Greedy best first search uses a heuristic(h(n)) which is defined as a number of mismatched neighboring domino edges in a state. For every tile that has a neighbor above it, the tile's top value is compared with the bottom value of the tile above. For every tile that has a neighbor to its left, the tile's left value is compared with the right value of the neighboring tile. Each mismatch adds one to the heuristic value. Therefore, a smaller heuristic value represents a board with more correctly matching edges, while a goal state has h(n) = 0. GBFS uses this value to prioritize states that appear closer to a valid solution.

# Breadth-First Search(BFS) is used as an uninformed search algorithm. It stores discovered states in a FIFO queue and explores the search space by each level. In the project, each level corresponds to an additional tile swap from the initial state Because every swap has the same cost, BFS finds a solution using the minimum number of swaps if it finds a solution before the expansion limit is reached. A visited set is used to prevent duplicate board arrangements from being explored more than once.

# Greedy Best-First Search (GBFS): GBFS is used as the informed search algorithm. Instead of expanding states in the order they were discovered, it uses a priority queue and selects the state with the smallest heuristic value h(n). In this project, h(n) is the number of mismatched neighboring domino edges. Therefore, GBFS prioritizes arrangements that appear closer to a solved puzzle. Unlike BFS, GBFS does not guarantee the shortest solution path because it considers only the heuristic estimate and not the number of swaps already taken.

# Experimental Setup: BFS and GBFS were compared on puzzle sizes N=2, N=3, and N=4. For each board size, 10 randomly generated initial states were tested. To ensure a fair comparison, BFS and GBFS were run on the exact same initial state for each trial. Both algorithms used a maximum expansion limit of 100,000 states. A trial was considered successful if the algorithm found a valid goal state before reaching this limit. For each algorithm and board size, the success rate and average number of expanded states were recorded.                                                                                                          For failed trials, the reported expansion count is the 100,000-state cutoff. Therefore, the BFS average of 100,000 states for N=4 does not mean that BFS normally solves the puzzle after exactly 100,000 expansions. It means that BFS reached the imposed search limit in all 10 trials without finding a solution.

# Results Analysis: The results show that both BFS and GBFS performed well on the small N=2 puzzle, but their performance diverged sharply as the puzzle size increased. For N=2, BFS solved all trials and expanded an average of 11.1 states, while GBFS also solved all trials but required only 3.3 expanded states on average. At this size, the search space is small enough that uninformed search remains practical. 

# For N=3, BFS performance decreased substantially. Its success rate fell to 20%, and it expanded an average of 89,748.8 states before either finding a solution or reaching the 100,000-state cutoff. In contrast, GBFS maintained a 100% success rate and expanded an average of only 84.5 states. This difference occurs because BFS explores states level by level without considering whether a board arrangement is becoming more promising, while GBFS uses the heuristic to prioritize states with fewer mismatched edges.

# The difference became even more significant for N=4. BFS failed to find a solution in all 10 trials before reaching the 100,000-state limit. GBFS, however, solved all 10 trials while expanding an average of 1,765.8 states. Although GBFS also required more work as the board size increased, the heuristic allowed it to search a much smaller and more promising portion of the state space.

# These results demonstrate the effect of the rapidly growing state space and branching factor. As N increases, both the number of possible tile arrangements and the number of possible swaps from each state increase dramatically. BFS does not have information about which states are closer to a solution, so much of its work is spent exploring unpromising arrangements. GBFS uses the mismatch heuristic to guide the search toward states that contain more correctly matched neighboring edges, which greatly reduces the number of states that need to be expanded in these experiments.

# GBFS was more efficient in these experiments, but it is not guaranteed to find the shortest solution path. BFS does have that guarantee here when it completes, because every swap has equal cost.

# Limitations: The experiments used a maximum expansion limit of 100,000 states, so a failed trial does not prove that an algorithm is incapable of finding a solution. It only means that the solution was not found within the allowed search budget. The heuristic used by GBFS is also relatively simple: it counts mismatched neighboring edges but does not consider how many swaps may actually be required to repair those mismatches. As a result, a state with a low heuristic value may still be far from a goal state. In addition, GBFS does not guarantee the shortest solution path because it prioritizes only the heuristic value and ignores the number of moves already taken.

# Conclusion: This project compared an uninformed search strategy, Breadth-First Search, with an informed strategy, Greedy Best-First Search, on the four-sided domino puzzle. Both algorithms performed successfully on small 2 × 2 instances, but BFS became much less practical as the puzzle size increased because of the factorial growth of the state space and the increasing branching factor. In the experiments, GBFS maintained a 100% success rate for all tested board sizes and expanded substantially fewer states than BFS by using the mismatch heuristic to guide its search. The results illustrate how heuristic information can greatly improve search efficiency in large combinatorial problems, while also showing the tradeoff that GBFS does not guarantee an optimal solution path.

### Final Results

| N | BFS Success Rate | BFS Avg. Expanded | GBFS Success Rate | GBFS Avg. Expanded |
|---|---:|---:|---:|---:|
| 2 | 100% | 11.1 | 100% | 3.3 |
| 3 | 20% | 89,748.8 | 100% | 84.5 |
| 4 | 0% | 100,000.0 | 100% | 1,765.8 |

For failed trials, the expansion count is the 100,000-state cutoff. Therefore, the BFS value of 100,000 for N=4 indicates that BFS reached the cutoff in all 10 trials without finding a solution.